# Ontology Agent Evaluation with Strands Framework

This notebook evaluates the Ontology Generation Agent using the AWS Strands evaluation framework.

## AWS Credentials Setup

The ontology agent needs AWS credentials to access:
- **Amazon Bedrock** - For Claude model inference (`global.anthropic.claude-sonnet-4-6`)
- **AWS Glue / Athena** - For retrieving database schemas (`get_single_table_schema`, `sample_table_data`)
- **Amazon Neptune** - For persisting N-Quads (via AgentCore Gateway — `persist_file_to_neptune`)
- **Amazon S3** - For storing per-table N-Quads files and reference documents
- **Amazon Bedrock Knowledge Base** - For retrieving ontology patterns (`retrieve_ontology_patterns`)
- **Amazon DynamoDB** - For reading config and writing status updates (`ONTOLOGY_METADATA_TABLE`)

### Credential Injection

This notebook uses an AWS profile configured locally. The boto3 session is created with this profile and then injected into the ontology agent using `set_boto_session()`. This ensures all AWS service calls within the agent use the same credentials.

### Production Code Path

This notebook exercises the **full production entry point** — the same code path as the deployed Lambda + AgentCore runtime:

1. A DynamoDB record (type `"VKG"`) is created with `status="draft"` — matching what `OntologyService.create_metadata_config()` writes
2. `invoke(payload={"id": ontology_id}, context={})` is called — the same entrypoint Lambda invokes via `AgentCoreService.invoke_ontology_agent()`
3. `invoke()` reads config from DynamoDB, updates status to `"processing"`, and starts the background thread (Phase 1 → Phase 2 → Assembly)
4. The notebook polls DynamoDB until `status` reaches `"completed"` or `"failed"` before capturing Strands spans

### Required Environment Variables
- `KNOWLEDGE_BASE_ID` — Bedrock KB for ontology pattern retrieval
- `ARTIFACTS_BUCKET` — S3 bucket for N-Quads file staging and reference docs
- `NEPTUNE_GATEWAY_URL` — AgentCore Gateway endpoint for Neptune persistence (Phase 2)
- `ONTOLOGY_METADATA_TABLE` — DynamoDB table for ontology metadata / status tracking
- `AWS_REGION` — AWS region (default: `us-east-1`)

In [1]:
# Install required packages for evaluation
# !pip install strands-agents strands-agents-tools strands-agents-evals pandas --quiet

In [2]:
# Verify installations
try:
    import strands
    import strands_evals
    print("✓ strands-agents installed:", strands.__version__ if hasattr(strands, '__version__') else "version unknown")
    print("✓ strands-evals installed:", strands_evals.__version__ if hasattr(strands_evals, '__version__') else "version unknown")
except ImportError as e:
    print(f"✗ Import error: {e}")


✓ strands-agents installed: version unknown
✓ strands-evals installed: version unknown


In [3]:
# Import required libraries
import sys
import os
import json
import uuid
import time
import pandas as pd
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from datetime import datetime, timezone

# Add ontology agent to path
sys.path.append('../agents')

# Import agent entry point
from ontology_agent.main import invoke, set_boto_session

print("✓ Imports successful")

✓ Imports successful


In [4]:
# Load environment variables from .env file  (which have been sourced from cdk/cdk-outputs-agentcore.json)

from dotenv import load_dotenv

load_dotenv(dotenv_path='.env', override=False)

os.environ['KNOWLEDGE_BASE_ID']        = os.environ.get('KNOWLEDGE_BASE_ID', 'XXX')
os.environ['ARTIFACTS_BUCKET']         = os.environ.get('ARTIFACTS_BUCKET', 'XXX')
os.environ['NEPTUNE_GATEWAY_URL']      = os.environ.get('NEPTUNE_GATEWAY_URL', 'XXX')
os.environ['NEPTUNE_LOAD_ROLE']        = os.environ.get('NEPTUNE_LOAD_ROLE', 'XXX')
os.environ['ONTOLOGY_METADATA_TABLE']  = os.environ.get('ONTOLOGY_METADATA_TABLE', 'XXX')
os.environ['ATHENA_WORKGROUP']         = os.environ.get('ATHENA_WORKGROUP', 'XXX')

print("✓ Environment variables configured:")
for key in ['AWS_PROFILE', 'AWS_DEFAULT_REGION', 'KNOWLEDGE_BASE_ID', 'ARTIFACTS_BUCKET',
            'NEPTUNE_GATEWAY_URL', 'NEPTUNE_LOAD_ROLE', 'ONTOLOGY_METADATA_TABLE', 'ATHENA_WORKGROUP']:
    print(f"  {key} = {os.environ.get(key, '(not set)')}")

✓ Environment variables configured:
  AWS_PROFILE = huthmac-demo
  AWS_DEFAULT_REGION = us-east-1
  KNOWLEDGE_BASE_ID = L9EZ5SAY3F
  ARTIFACTS_BUCKET = semantic-layer-artifacts-381492284087
  NEPTUNE_GATEWAY_URL = https://semantic-layer-neptune-gateway-pt7i4au20m.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
  NEPTUNE_LOAD_ROLE = arn:aws:iam::381492284087:role/semantic-layer-neptune-NeptuneS3LoadRole2467807B-52cfATOqGIS4
  ONTOLOGY_METADATA_TABLE = semantic-layer-metadata
  ATHENA_WORKGROUP = semantic-layer-workgroup


In [5]:
config = Config(
    retries={'max_attempts': 10, 'mode': 'standard'},
    signature_version='v4'
)

# Create session with specific AWS profile
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'XXX'))
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ['AWS_PROFILE']}")
    print(f"  Account: ...{identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

# Inject session into ontology agent
set_boto_session(session)
print(f"\n✓ Agent configured to use boto3 session credentials")
print(f"  All AWS service calls (Bedrock, Glue, Neptune, S3, Knowledge Base) will use this session")

2026-03-16 09:14:24,975 - ontology_agent.main - INFO - Boto3 session set with region: us-east-1


✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1

✓ Agent configured to use boto3 session credentials
  All AWS service calls (Bedrock, Glue, Neptune, S3, Knowledge Base) will use this session


## Define Evaluation Test Cases

Test cases target **S3 Tables (Iceberg)** in the `semantic_layer_iceberg` namespace of bucket `semantic-layer-analytics-tables`. This avoids Lake Formation access issues that affect DynamoDB-backed Glue tables when running under the cross-account Isengard `Admin` role.

The Athena catalog ID `s3tablescatalog/semantic-layer-analytics-tables` routes queries directly through the S3 Tables catalog — no LF permission checks apply.

Each test case includes:
- **id**: Unique test identifier
- **category**: `single_table` or `multi_table`
- **input_config**: Ontology config with `dataSources` pointing to S3 Tables
- **expected_outcomes**: Expected results to validate
- **validation_criteria**: Specific checks to perform

In [6]:
# S3 Tables (Iceberg) data source config
# Bucket:    semantic-layer-analytics-tables
# Namespace: semantic_layer_iceberg
# CatalogId: s3tablescatalog/semantic-layer-analytics-tables  (Athena S3 Tables catalog)
S3T_CATALOG  = os.environ.get('S3T_CATALOG', 'XXX')
S3T_DATABASE = os.environ.get('S3T_DATABASE', 'XXX')

DYNAMODB_CATALOG  = os.environ.get('DYNAMODB_CATALOG', 'XXX')
DYNAMODB_DATABASE = os.environ.get('DYNAMODB_DATABASE', 'XXX')

# Timestamp embedded in DynamoDB name field so eval runs are easy to find/sort
RUN_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M')

def s3t(table_name: str) -> dict:
    return {
        'databaseName': S3T_DATABASE,
        'tableName':    table_name,
        'catalogId':    S3T_CATALOG,
        'dataSource':   'AwsDataCatalog',   # S3 Tables uses Athena native catalog — matches UI logic
        'tableId':      f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

def dynamot(table_name: str, glue_database: str = "", glue_table: str = "") -> dict:
    entry = {
        'databaseName': DYNAMODB_DATABASE,
        'tableName':    table_name,
        'catalogId':    DYNAMODB_CATALOG,
        'dataSource':   DYNAMODB_CATALOG,
        'tableId':      f'{DYNAMODB_CATALOG}::{DYNAMODB_DATABASE}.{table_name}',
    }
    if glue_database:
        entry['glueDatabaseName'] = glue_database
    if glue_table:
        entry['glueTableName'] = glue_table
    return entry

test_cases = [
    # {
    #     'id': 'single_table_basic',
    #     'ontology_id': 'vkg-test-001',
    #     'category': 'single_table_s3tables',
    #     'dynamodb_config': {
    #         'name': f'admincodes-s3tables {RUN_TIMESTAMP}',
    #         'type': 'VKG',
    #         'dataSourcesDescription': 'Iceberg S3 Tables in semantic_layer_iceberg namespace',
    #         'useCasesDescription': 'Enable business users to query administrative codes',
    #         'dataSources': [s3t('admincode')],
    #     },
    #     'expected_outcomes': {
    #         'classes_created': ['admincode'],
    #         'properties_per_table': 'all_columns',
    #         'traceability_mappings': True,
    #         'neptune_persistence': True,
    #     },
    #     'validation_criteria': [
    #         'All columns mapped to properties',
    #         'mapsToTable predicate present',
    #         'mapsToColumn predicates present',
    #         'Valid N-Quads format',
    #         'Named graph matches namespace',
    #     ],
    # },
    {
        'id': 'single_table_basic',
        'ontology_id': 'vkg-test-002',
        'category': 'single_table_dynamodb',
        'dynamodb_config': {
            'name': f'admincodes-dyndb {RUN_TIMESTAMP}',
            'type': 'VKG',
            'dataSourcesDescription': 'DynamoDB Tables directly',
            'useCasesDescription': 'Enable business users to query administrative codes',
            'dataSources': [dynamot(
                'semantic-layer-admin-codes',
                glue_database='semantic_layer_dynamodb',
                glue_table='semantic_layer_admin_codes',
            )],
        },
        'expected_outcomes': {
            'classes_created': ['semantic-layer-admin-codes'],
            'properties_per_table': 'all_columns',
            'traceability_mappings': True,
            'neptune_persistence': True,
        },
        'validation_criteria': [
            'All columns mapped to properties',
            'mapsToTable predicate present',
            'mapsToColumn predicates present',
            'Valid N-Quads format',
            'Named graph matches namespace',
        ],
    },
    # {
    #     'id': 'multi_table_relationships',
    #     'ontology_id': 'eval-test-002',
    #     'category': 'multi_table',
    #     'dynamodb_config': {
    #         'name': f'insurance-policy {RUN_TIMESTAMP}',
    #         'type': 'VKG',
    #         'dataSourcesDescription': 'Iceberg S3 Tables in semantic_layer_iceberg namespace',
    #         'useCasesDescription': 'Insurance policies, coverages, and holdings ontology',
    #         'dataSources': [s3t('policyproduct'), s3t('coverage'), s3t('holding')],
    #     },
    #     'expected_outcomes': {
    #         'classes_created': ['PolicyProduct', 'Coverage', 'Holding'],
    #         'foreign_keys_as_object_properties': True,
    #         'traceability_mappings': True,
    #         'neptune_persistence': True,
    #     },
    #     'validation_criteria': [
    #         'FK hints recorded for all tables',
    #         'All tables have mapsToTable',
    #         'Consistent namespace usage',
    #         'Incremental append_nquads calls (10-column batches)',
    #     ],
    # },
]

print(f"✓ Defined {len(test_cases)} test cases (run timestamp: {RUN_TIMESTAMP})")
print(f"\nCategories:")
categories = {}
for tc in test_cases:
    cat = tc['category']
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} test case(s)")
print(f"\nTables used:")
for tc in test_cases:
    tables = [ds['tableName'] for ds in tc['dynamodb_config']['dataSources']]
    print(f"  {tc['id']}: {', '.join(tables)}")
    print(f"  DynamoDB name: \"{tc['dynamodb_config']['name']}\"")

✓ Defined 1 test cases (run timestamp: 2026-03-16 09:14)

Categories:
  single_table_dynamodb: 1 test case(s)

Tables used:
  single_table_basic: semantic-layer-admin-codes
  DynamoDB name: "admincodes-dyndb 2026-03-16 09:14"


## Setup Strands Evaluation Framework

Configure telemetry and evaluation infrastructure for capturing agent execution traces

In [7]:
from strands_evals import Case, Experiment
from strands_evals.evaluators import (
    HelpfulnessEvaluator,
    FaithfulnessEvaluator,
    ToolSelectionAccuracyEvaluator,
    GoalSuccessRateEvaluator,
)
from strands_evals.mappers import StrandsInMemorySessionMapper
from strands_evals.telemetry import StrandsEvalsTelemetry

# Setup telemetry for capturing agent spans
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print("✓ Strands evaluation framework initialized")
print(f"  Telemetry: In-memory exporter configured")
print(f"  Ready to capture agent traces")

2026-03-16 09:14:24,998 - strands_evals.telemetry.config - INFO - Initializing tracer for strands-evals
2026-03-16 09:14:24,999 - strands_evals.telemetry.config - INFO - Enabling in-memory export for strands-evals


✓ Strands evaluation framework initialized
  Telemetry: In-memory exporter configured
  Ready to capture agent traces


## Convert Test Cases to Strands Format

Transform the test dataset into Strands `Case` objects with structured prompts that mirror the production invocation payload.

In [8]:
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')

def seed_test_ontology(
    ontology_id: str,
    tables: list,
    name: str = 'vkg-eval-test',
    data_sources_description: str = 'vkg test',
    use_cases_description: str = 'vkg test',
    created_by: str = 'eval@semantic-layer.local',
) -> None:
    """
    Write a DynamoDB item matching the shape created by the admin UI flow
    (OntologyService.create_metadata_config + start_build_metadata_async).
    Each entry in `tables` is a full data-source dict from s3t() / dynamot().
    """
    dynamodb = session.resource('dynamodb')
    ddb_table = dynamodb.Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    data_sources = [
        {
            'databaseName': t['databaseName'],
            'tableName':    t['tableName'],
            'catalogId':    t.get('catalogId', 'AWSDataCatalog'),
            'dataSource':   t.get('dataSource', 'AwsDataCatalog'),
            'tableId':      t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]

    item = {
        'id':             ontology_id,
        'version':        'v1',
        'type':           'VKG',
        'name':           name,
        'status':         'pending',
        'configuration':  {},
        'dataSources':    data_sources,
        'createdAt':      now,
        'updatedAt':      now,
        'buildStartedAt': now,
        'createdBy':      created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description

    ddb_table.put_item(Item=item)
    print(f"✓ Seeded DynamoDB item: {ontology_id} ({len(data_sources)} table(s))")

print("✓ seed_test_ontology defined")


strands_cases = []

for test_case in test_cases:
    job_id = f"{test_case['id']}-{uuid.uuid4().hex[:8]}"
    cfg = test_case['dynamodb_config']
    tables = cfg['dataSources']

    table_names = ', '.join(t['tableName'] for t in tables)
    prompt = (
        f"Generate an OWL ontology (N-Quads) for "
        f"{len(tables)} table(s): {table_names}. Ontology ID: {job_id}."
    )
    expected_output = (
        f"OWL ontology generated for: {table_names}. "
        f"Each table has an owl:Class with owl:DatatypeProperty instances per column, "
        f"mapsToTable / mapsToColumn traceability predicates, and N-Quads saved to S3 "
        f"and persisted to Neptune."
    )

    case = Case[str, str](
        name=test_case['id'],
        input=prompt,
        expected_output=expected_output,
        metadata={
            'category': test_case['category'],
            'job_id': job_id,
            'tables': tables,
            'dynamodb_config': {
                'name': cfg.get('name', 'eval-test'),
                'dataSourcesDescription': cfg.get('dataSourcesDescription', ''),
                'useCasesDescription': cfg.get('useCasesDescription', ''),
            },
            'expected_outcomes': test_case['expected_outcomes'],
            'validation_criteria': test_case['validation_criteria'],
        }
    )
    strands_cases.append(case)

print(f"✓ Converted {len(strands_cases)} test case(s) to Strands Case format")
print(f"\nCases:")
for case in strands_cases:
    tables = [t['tableName'] for t in case.metadata['tables']]
    print(f"  {case.name} ({case.metadata['category']}): {', '.join(tables)}")

# Store job_id for the query evaluation notebook to pick up via %store -r job_id
%store job_id
print(f"\n✓ job_id stored: {job_id}")

✓ seed_test_ontology defined
✓ Converted 1 test case(s) to Strands Case format

Cases:
  single_table_basic (single_table_dynamodb): semantic-layer-admin-codes
Stored 'job_id' (str)

✓ job_id stored: single_table_basic-d7b2932e


## Define Task Function

Create task function that executes the ontology agent and captures telemetry

In [9]:
def ontology_agent_task(case: Case) -> dict:
    memory_exporter.clear()
    tables = case.metadata['tables']
    job_id = case.metadata['job_id']
    cfg = case.metadata.get('dynamodb_config', {})

    # Seed DynamoDB — mirrors what the admin UI flow creates before invoking the agent
    seed_test_ontology(
        job_id,
        tables,
        name=cfg.get('name', 'eval-test'),
        data_sources_description=cfg.get('dataSourcesDescription', ''),
        use_cases_description=cfg.get('useCasesDescription', ''),
    )

    try:
        start_time = datetime.now()
        failures = []

        # Invoke the ontology agent — same payload Lambda sends via AgentCore.
        # invoke() reads DynamoDB, updates status to 'processing', and starts
        # the background thread (Phase 1 → Phase 2 → Assembly).
        print(f"[{case.name}] Invoking ontology agent for: {job_id}")
        invoke(payload={"id": job_id}, context={})

        # Poll DynamoDB until the background thread finishes
        dynamodb_r = session.resource('dynamodb')
        ddb_table = dynamodb_r.Table(METADATA_TABLE)
        max_wait_secs = 600   # 10 minutes
        poll_interval = 15    # seconds

        print(f"[{case.name}] Polling DynamoDB for completion (max {max_wait_secs}s)...")
        final_item = {}
        final_status = 'processing'
        while (datetime.now() - start_time).total_seconds() < max_wait_secs:
            time.sleep(poll_interval)
            resp = ddb_table.get_item(Key={'id': job_id, 'version': 'v1'})
            final_item = resp.get('Item', {})
            final_status = final_item.get('status', 'unknown')
            elapsed = (datetime.now() - start_time).total_seconds()
            print(f"  [{elapsed:.0f}s] status={final_status}")
            if final_status in ('completed', 'failed'):
                break

        duration = (datetime.now() - start_time).total_seconds()

        # Build summary from final DynamoDB state
        if final_status == 'completed':
            tables_done = final_item.get('tablesProcessed', len(tables))
            total = final_item.get('totalTables', len(tables))
            summary = f"Ontology build completed: {tables_done}/{total} table(s) processed."
        elif final_status == 'failed':
            error_msg = final_item.get('error', 'unknown error')
            failures.append(error_msg)
            summary = f"Ontology build failed: {error_msg}"
        else:
            failures.append(f"Timed out after {max_wait_secs}s with status: {final_status}")
            summary = f"Ontology build timed out. Final status: {final_status}"

        print(f"✓ {summary}")

        finished_spans = list(memory_exporter.get_finished_spans())
        mapper = StrandsInMemorySessionMapper()
        session_obj = mapper.map_to_session(finished_spans, session_id=case.session_id)

        return {
            'output':     summary,
            'trajectory': session_obj,
            'duration':   duration,
            'success':    len(failures) == 0,
            'error':      ', '.join(failures) if failures else None,
        }

    except Exception as e:
        print(f"✗ Error in {case.name}: {str(e)}")
        import traceback; traceback.print_exc()
        return {'output': '', 'trajectory': None, 'duration': 0, 'success': False, 'error': str(e)}

print("✓ ontology_agent_task defined")

✓ ontology_agent_task defined


## Create Built-in Evaluators

Configure Strands evaluators for multi-dimensional assessment of ontology generation

In [10]:
from strands.models import BedrockModel

# Create judge model using same boto session
judge_model = BedrockModel(
    model_id='global.anthropic.claude-sonnet-4-6',
    temperature=0.0,
    boto_session=session
)

# Initialize built-in evaluators
evaluators = [
    # TRACE_LEVEL: Response quality evaluation
    HelpfulnessEvaluator(model=judge_model),

    # TRACE_LEVEL: Faithfulness to context (detects hallucinations)
    FaithfulnessEvaluator(model=judge_model),

    # TOOL_LEVEL: Validate tool selection (Glue, Neptune, S3, RAG)
    ToolSelectionAccuracyEvaluator(model=judge_model),

    # SESSION_LEVEL: Overall goal achievement
    GoalSuccessRateEvaluator(model=judge_model)
]

print(f"✓ Created {len(evaluators)} evaluators:")
print(f"  1. HelpfulnessEvaluator  - Ontology quality & completeness")
print(f"  2. FaithfulnessEvaluator - Schema accuracy & hallucination detection")
print(f"  3. ToolSelectionAccuracyEvaluator - Correct tool usage")
print(f"  4. GoalSuccessRateEvaluator - Overall success rate")
print(f"\n  All evaluators use claude-sonnet-4-6 as judge")

✓ Created 4 evaluators:
  1. HelpfulnessEvaluator  - Ontology quality & completeness
  2. FaithfulnessEvaluator - Schema accuracy & hallucination detection
  3. ToolSelectionAccuracyEvaluator - Correct tool usage
  4. GoalSuccessRateEvaluator - Overall success rate

  All evaluators use claude-sonnet-4-6 as judge


## Run Evaluation Experiment

Execute test cases and evaluate with all built-in evaluators

In [11]:
# Create experiment with test cases and evaluators
experiment = Experiment[str, str](
    cases=strands_cases,
    evaluators=evaluators
)

print(f"Starting experiment with {len(strands_cases)} test cases...")
print(f"{'='*70}\\n")

# Run evaluations
reports = experiment.run_evaluations(ontology_agent_task)

print(f"\\n{'='*70}")
print(f"✓ Evaluation complete!")
print(f"  Generated {len(reports)} evaluation reports")

Starting experiment with 1 test cases...
======================================================================\n


2026-03-16 09:14:25,332 - ontology_agent.main - INFO - [Entrypoint] Starting ontology build for: single_table_basic-d7b2932e


✓ Seeded DynamoDB item: single_table_basic-d7b2932e (1 table(s))
[single_table_basic] Invoking ontology agent for: single_table_basic-d7b2932e


2026-03-16 09:14:25,547 - ontology_agent.main - INFO - [Entrypoint] Loaded config (version=v1): admincodes-dyndb 2026-03-16 09:14 with 1 data sources (namespace: http://admincodes-dyndb-2026-03-16-09-14/ontology/single_table_basic-d7b2932e)
2026-03-16 09:14:25,727 - ontology_agent.main - INFO - Updated DynamoDB: single_table_basic-d7b2932e (version=v1) → processing
2026-03-16 09:14:25,729 - ontology_agent.main - INFO - [Entrypoint] 1 tables to process
/Users/huthmac/Documents/AWS/00_workspace/semantic-layer/venv/lib/python3.12/site-packages/bedrock_agentcore/runtime/app.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
{"timestamp": "2026-03-16T13:14:25.734Z", "level": "INFO", "message": "Async task started: ontology_build (ID: -8012997158535300575)", 

[single_table_basic] Polling DynamoDB for completion (max 600s)...


...

Neptune persistence succeeded. Now updating Glue metadata:
Tool #2: update_glue_metadata_from_ontology


2026-03-16 09:17:32,358 - ontology_agent.main - WARNING - DynamoDB table semantic-layer-admin-codes is not registered in the Glue Data Catalog (connector catalog: dynamodb_catalog). Glue metadata not updated.


**Summary:** `semantic-layer-admin-codes` — ObjectProperties added: **0** (no FKs) — Neptune: ✅ **success** (85 triples persisted) — Glue: ⚠️ skipped (table not registered in Glue Data Catalog via DynamoDB connector — expected, not an error).

2026-03-16 09:17:35,932 - ontology_agent.main - INFO - [Phase 2] Refinement complete for single_table_basic-d7b2932e
2026-03-16 09:17:35,933 - ontology_agent.main - INFO - [Assembly] Building consolidated ontology for single_table_basic-d7b2932e
2026-03-16 09:17:36,302 - ontology_agent.main - INFO - ✓ Successfully saved ontology to S3: s3://semantic-layer-artifacts-381492284087/ontologies/single_table_basic-d7b2932e/ontology.nq
2026-03-16 09:17:36,303 - ontology_agent.main - INFO - [Assembly] Saved consolidated ontology: s3://semantic-layer-artifacts-381492284087/ontologies/single_table_basic-d7b2932e/ontology.nq
2026-03-16 09:17:36,609 - ontology_agent.main - INFO - Updated DynamoDB: single_table_basic-d7b2932e (version=v1) → processing
2026-03-16 09:17:36,610 - ontology_agent.main - INFO - [Iceberg] Writing column doc strings and table descriptions for single_table_basic-d7b2932e
2026-03-16 09:17:36,979 - ontology_agent.main - INFO - [Iceberg] No S3 Tables in this ontology — skipping

  [196s] status=completed
✓ Ontology build completed: 1/1 table(s) processed.
\n======================================================================
✓ Evaluation complete!
  Generated 4 evaluation reports


## Aggregate Metrics

Analyze scores across all test cases and evaluators

In [12]:
# Aggregate evaluation scores across all test cases
# reports is list[EvaluationReport] — one report per evaluator, each report
# has parallel lists indexed by test case: scores[i], test_passes[i], cases[i]

evaluation_data = []

n_cases = len(reports[0].cases) if reports else 0

for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = (case_dict.get('metadata') or {})
    case_data = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
        'scope': metadata.get('scope', 'unknown'),
    }

    # Each report corresponds to one evaluator (same order as evaluators list)
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        case_data[eval_name] = report.scores[i] if i < len(report.scores) else None

    evaluation_data.append(case_data)

# Create DataFrame for analysis
df_evals = pd.DataFrame(evaluation_data)

print("=== EVALUATION SCORES BY TEST CASE ===\n")
print(df_evals.to_string(index=False))

evaluator_cols = [col for col in df_evals.columns if col not in ['test_id', 'category', 'scope']]

print("\n=== AVERAGE SCORES BY EVALUATOR ===\n")
avg_scores = df_evals[evaluator_cols].mean()
print(avg_scores.to_string())

print("\n=== SCORES BY CATEGORY ===\n")
category_scores = df_evals.groupby('category')[evaluator_cols].mean()
print(category_scores.to_string())

=== EVALUATION SCORES BY TEST CASE ===

           test_id              category   scope  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
single_table_basic single_table_dynamodb unknown                   1.0                    1.0                             1.0                       1.0

=== AVERAGE SCORES BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== SCORES BY CATEGORY ===

                       HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                                    
single_table_dynamodb                   1.0                    1.0                             1.0                       1.0


## Pass/Fail Analysis

Identify which test cases passed or failed each evaluator

In [13]:
# Analyze pass/fail status for each evaluator
pass_fail_data = []

n_cases = len(reports[0].cases) if reports else 0

for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = (case_dict.get('metadata') or {})
    case_pf = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
    }

    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        case_pf[eval_name] = report.test_passes[i] if i < len(report.test_passes) else None

    pass_fail_data.append(case_pf)

df_pass_fail = pd.DataFrame(pass_fail_data)

print("=== PASS/FAIL BY TEST CASE ===\n")
print(df_pass_fail.to_string(index=False))

evaluator_cols = [col for col in df_pass_fail.columns if col not in ['test_id', 'category']]

print("\n=== PASS RATE BY EVALUATOR ===\n")
pass_rates = df_pass_fail[evaluator_cols].mean()
print(pass_rates.to_string())

print("\n=== PASS RATE BY CATEGORY ===\n")
category_pass_rates = df_pass_fail.groupby('category')[evaluator_cols].mean()
print(category_pass_rates.to_string())

=== PASS/FAIL BY TEST CASE ===

           test_id              category  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
single_table_basic single_table_dynamodb                  True                   True                            True                      True

=== PASS RATE BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== PASS RATE BY CATEGORY ===

                       HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                                    
single_table_dynamodb                   1.0                    1.0                             1.0                       1.0


## Display Detailed Report

View comprehensive evaluation metrics for a sample test case

In [14]:
# Display detailed report for each evaluator using Rich inline rendering
if reports:
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        print(f"\n=== {eval_name} ===")
        report.display()
else:
    print("No reports generated")


=== HelpfulnessEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘


=== FaithfulnessEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘


=== GoalSuccessRateEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘

## Save Evaluation Results

Export detailed evaluation data for further analysis

In [15]:
# Save detailed evaluation results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save scores DataFrame
scores_file = f'../data/eval/results/ontology_evaluation_scores_{timestamp}.csv'
df_evals.to_csv(scores_file, index=False)
print(f"✓ Scores saved to: {scores_file}")

# Save pass/fail DataFrame
pass_fail_file = f'../data/eval/results/ontology_evaluation_pass_fail_{timestamp}.csv'
df_pass_fail.to_csv(pass_fail_file, index=False)
print(f"✓ Pass/fail data saved to: {pass_fail_file}")

# Save detailed JSON — structured per test case, with all evaluator results
n_cases = len(reports[0].cases) if reports else 0
detailed_results = []

for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = (case_dict.get('metadata') or {})
    case_result = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
        'scope': metadata.get('scope', 'unknown'),
        'input': str(case_dict.get('input', ''))[:200] + '...',
        'expected': case_dict.get('expected_output', ''),
        'evaluations': {},
    }

    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        raw_detailed = report.detailed_results[i] if i < len(report.detailed_results) else []
        eval_outputs = [
            {'score': r.score, 'test_pass': r.test_pass, 'label': r.label, 'reasoning': r.reason}
            for r in raw_detailed
        ] if isinstance(raw_detailed, list) else []

        case_result['evaluations'][eval_name] = {
            'score': report.scores[i] if i < len(report.scores) else None,
            'test_pass': report.test_passes[i] if i < len(report.test_passes) else None,
            'reason': report.reasons[i] if i < len(report.reasons) else '',
            'detailed': eval_outputs,
        }

    detailed_results.append(case_result)

detailed_file = f'../data/eval/results/ontology_evaluation_detailed_{timestamp}.json'
with open(detailed_file, 'w') as f:
    json.dump(detailed_results, f, indent=2)
print(f"✓ Detailed results with reasoning saved to: {detailed_file}")

print(f"\n{'='*70}")
print(f"✓ All evaluation results saved")
print(f"  Scores: {scores_file}")
print(f"  Pass/Fail: {pass_fail_file}")
print(f"  Detailed: {detailed_file}")

✓ Scores saved to: ../data/eval/results/ontology_evaluation_scores_20260316_092059.csv
✓ Pass/fail data saved to: ../data/eval/results/ontology_evaluation_pass_fail_20260316_092059.csv
✓ Detailed results with reasoning saved to: ../data/eval/results/ontology_evaluation_detailed_20260316_092059.json

✓ All evaluation results saved
  Scores: ../data/eval/results/ontology_evaluation_scores_20260316_092059.csv
  Pass/Fail: ../data/eval/results/ontology_evaluation_pass_fail_20260316_092059.csv
  Detailed: ../data/eval/results/ontology_evaluation_detailed_20260316_092059.json


## Summary

### Agent Architecture (Phase 1)
The Phase 1 ontology agent generates OWL ontologies from Athena/Glue table schemas:

1. **`get_single_table_schema(database, table, catalog_id)`** — fetches column definitions via Athena DESCRIBE TABLE (with Glue fallback for DynamoDB-backed tables)
2. **`sample_table_data(database, table, catalog_id)`** — retrieves sample rows to confirm FK patterns and enrich `rdfs:comment` values
3. **`retrieve_ontology_patterns(schema_description)`** — fetches relevant OWL patterns from Bedrock Knowledge Base
4. **`append_nquads(ontology_id, table_name, nquad_batch)`** — writes N-Quads incrementally (max 10 columns / ~70 lines per call)
5. **`save_intermediate_ontology(..., fk_hints=...)`** — finalises per-table N-Quads and records FK hints for Phase 2
6. **`update_progress(...)`** — writes progress to DynamoDB

### Multi-Dimensional Evaluation
- **HelpfulnessEvaluator**: Ontology quality and completeness (7-level scale)
- **FaithfulnessEvaluator**: Schema accuracy and hallucination detection (5-level scale)
- **ToolSelectionAccuracyEvaluator**: Correct tool usage (Glue/Athena → RAG → append_nquads)
- **GoalSuccessRateEvaluator**: End-to-end Phase 1 success rate

### Ontology-Specific Validation
1. **Schema Mapping Accuracy**: All database columns mapped to `owl:DatatypeProperty` instances
2. **Traceability Mappings**: `mapsToTable` and `mapsToColumn` predicates present on every class/property
3. **Batching Compliance**: `append_nquads` called in ≤10-column batches (N-Quads lines ≤70 per call)
4. **FK Hints**: FK relationships recorded in `save_intermediate_ontology(fk_hints=...)`
5. **Format Compliance**: Valid N-Quads (full URIs, no prefixes, named graph, trailing ` .`)

### Next Steps
- Review failed test cases to understand agent weaknesses
- Compare scores across categories (single_table vs multi_table vs rag_enhanced)
- Run Phase 2 evaluation to validate FK ObjectProperty injection and Neptune persistence
- Track metrics over time as agent improves